In [1]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your Hugging Face token:  ········


Hugging Face token set successfully!


In [2]:
# Step 2: Load FLARE-CD test set and preprocess
from datasets import load_dataset, Dataset
import re

print("加载 FLARE-CD 数据集...")
ds_raw = load_dataset("ChanceFocus/flare-cd", split="test")
print(f"成功加载! 样本数: {len(ds_raw)}, 列: {ds_raw.column_names}")

# 定义标签类型
LABELS = ["O", "B-CAUSE", "I-CAUSE", "B-EFFECT", "I-EFFECT"]

def parse_tokens_and_labels(sample):
    """从数据集中解析 tokens 和 labels"""
    # 数据集已经提供了 token 和 label 字段
    tokens = sample.get("token", [])
    labels = sample.get("label", [])
    text = sample.get("text", "")
    
    # 如果没有预分的 tokens，从 text 分词
    if not tokens and text:
        tokens = text.split()
    
    return tokens, labels

def _map_row(x):
    tokens, labels = parse_tokens_and_labels(x)
    
    # 确保 tokens 和 labels 长度匹配
    if len(tokens) != len(labels):
        print(f"警告: tokens长度({len(tokens)}) != labels长度({len(labels)})")
        min_len = min(len(tokens), len(labels))
        tokens = tokens[:min_len]
        labels = labels[:min_len]
    
    return {
        "tokens": tokens,
        "labels": labels,
        "text": " ".join(tokens) if tokens else x.get("text", ""),
        "label_types": LABELS
    }

ds = Dataset.from_list([_map_row(r) for r in ds_raw])
print(f"处理完成! 样本数: {len(ds)}")

# 显示第一个样本
print("\n第一个样本示例:")
print(f"Tokens: {ds[0]['tokens'][:20]}...")
print(f"Labels: {ds[0]['labels'][:20]}...")

加载 FLARE-CD 数据集...
成功加载! 样本数: 226, 列: ['id', 'query', 'answer', 'text', 'label', 'token']
处理完成! 样本数: 226

第一个样本示例:
Tokens: ['Around', '21,000', 'employees', ',', '9,000', 'of', 'whom', 'are', 'employed', 'in', 'the', 'UK', ',', 'are', 'to', 'be', 'made', 'redundant', 'after', 'the']...
Labels: ['B-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'O', 'B-CAUSE']...


In [3]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform
from importlib.metadata import version, PackageNotFoundError
import requests
import re

MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

dataset_name = "flare_cd"
run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
save_dir = os.getcwd()
pred_path = os.path.join(save_dir, f"{run_tag}_predictions.csv")
meta_path = os.path.join(save_dir, f"{run_tag}_metadata.json")

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-cd",
    "split": "test",
    "label_types": LABELS,
    "model": MODEL,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini evaluation on FLARE-CD (Token-level Causal Annotation)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your OpenAI API key:  ········


Meta saved -> C:\Users\klofu\flare_cd_o3_mini_metadata.json
MODEL: o3-mini


In [4]:
# Step 3.5: 测试10个样本并输出结果
print("="*60)
print("测试10个样本 - Token-level 因果关系标注")
print("="*60)

def _make_cd_token_prompt(tokens):
    """创建 token-level 因果关系标注的提示词"""
    text = " ".join(tokens)
    return (
        "Task: Perform token-level causal annotation on the following text.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        "1. For each token, label it as:\n"
        "   - B-CAUSE: beginning of a cause phrase\n"
        "   - I-CAUSE: inside a cause phrase\n"
        "   - B-EFFECT: beginning of an effect phrase\n"
        "   - I-EFFECT: inside an effect phrase\n"
        "   - O: no causal role\n"
        "2. Return the labels as a JSON array in the same order as tokens\n"
        "3. Return ONLY the JSON array, no additional text\n\n"
        "Example output format:\n"
        '["B-CAUSE", "I-CAUSE", "O", "B-EFFECT", "I-EFFECT"]'
    )

def parse_token_labels_response(response_text: str, num_tokens: int):
    """解析模型返回的 token 标签"""
    response_text = response_text.strip()
    
    # 尝试提取 JSON 数组
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if match:
        try:
            labels = json.loads(match.group())
            if isinstance(labels, list) and len(labels) == num_tokens:
                return labels, True, None
            elif isinstance(labels, list):
                # 长度不匹配，尝试截断或填充
                if len(labels) > num_tokens:
                    labels = labels[:num_tokens]
                else:
                    labels = labels + ["O"] * (num_tokens - len(labels))
                return labels, True, None
        except json.JSONDecodeError as e:
            return None, False, f"JSON解析失败: {e}"
    return None, False, "未找到JSON数组"

def test_single_sample(tokens):
    """测试单个样本"""
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_cd_token_prompt(tokens)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a linguistic annotation expert. Respond only with valid JSON arrays."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        
        if response.status_code != 200:
            return None, False, f"API Error: {response.status_code}"
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return None, False, "No response"
            
        txt = result['choices'][0]['message']['content']
        pred_labels, success, error = parse_token_labels_response(txt, len(tokens))
        return pred_labels, success, error if not success else None
        
    except Exception as e:
        return None, False, str(e)

# 测试前10个样本
test_size = min(10, len(ds))
print(f"\n开始测试前 {test_size} 个样本...\n")

test_results = []

for i in range(test_size):
    sample = ds[i]
    tokens = sample["tokens"]
    true_labels = sample["labels"]
    
    pred_labels, success, error = test_single_sample(tokens)
    
    test_results.append({
        "index": i,
        "tokens_preview": " ".join(tokens[:10]) + ("..." if len(tokens) > 10 else ""),
        "token_count": len(tokens),
        "true_labels_preview": true_labels[:5] if len(true_labels) > 5 else true_labels,
        "pred_labels_preview": pred_labels[:5] if pred_labels and len(pred_labels) > 5 else pred_labels,
        "success": success,
        "error": error
    })
    
    print(f"样本 {i}: {'✅' if success else '❌'} | Tokens: {len(tokens)} | 预测前5: {test_results[-1]['pred_labels_preview']}")

# 输出详细结果
print("\n" + "="*80)
print("测试结果详情")
print("="*80)

for result in test_results:
    print(f"\n样本 {result['index']}:")
    print(f"  文本预览: {result['tokens_preview']}")
    print(f"  真实标签前5: {result['true_labels_preview']}")
    print(f"  预测标签前5: {result['pred_labels_preview']}")
    print(f"  状态: {'成功' if result['success'] else f'失败 - {result["error"]}'}")

# 统计
success_count = sum(1 for r in test_results if r["success"])
print(f"\n统计: 成功 {success_count}/{test_size} ({success_count/test_size*100:.1f}%)")

测试10个样本 - Token-level 因果关系标注

开始测试前 10 个样本...

样本 0: ✅ | Tokens: 31 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 1: ✅ | Tokens: 46 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 2: ✅ | Tokens: 81 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 3: ✅ | Tokens: 13 | 预测前5: ['B-CAUSE', 'I-CAUSE', 'I-CAUSE', 'I-CAUSE', 'I-CAUSE']
样本 4: ✅ | Tokens: 22 | 预测前5: ['B-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT']
样本 5: ✅ | Tokens: 37 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 6: ✅ | Tokens: 29 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 7: ✅ | Tokens: 17 | 预测前5: ['O', 'O', 'O', 'O', 'B-EFFECT']
样本 8: ✅ | Tokens: 22 | 预测前5: ['O', 'O', 'O', 'B-EFFECT', 'I-EFFECT']
样本 9: ✅ | Tokens: 49 | 预测前5: ['B-CAUSE', 'I-CAUSE', 'I-CAUSE', 'I-CAUSE', 'I-CAUSE']

测试结果详情

样本 0:
  文本预览: Around 21,000 employees , 9,000 of whom are employed in...
  真实标签前5: ['B-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT', 'I-EFFECT']
  预测标签前5: ['O', 'O', 'O', 'O', 'O']
  状态: 成功

样本 1:
  文本预览: REUTERS/Aly Song/File Photo ( Reuters ) - Tencent Holdings Ltd...
  真实标签前5

In [ ]:
# Step 3.6: 自动评估前10个样本
print("="*60)
print("自动评估前10个样本 - FLARE-CD")
print("="*60)

import json
import time
from sklearn.metrics import classification_report, accuracy_score

def _make_cd_token_prompt(tokens):
    """创建 token-level 因果关系标注的提示词"""
    text = " ".join(tokens)
    return (
        "Task: Perform token-level causal annotation on the following text.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        "1. For each token, label it as:\n"
        "   - B-CAUSE: beginning of a cause phrase\n"
        "   - I-CAUSE: inside a cause phrase\n"
        "   - B-EFFECT: beginning of an effect phrase\n"
        "   - I-EFFECT: inside an effect phrase\n"
        "   - O: no causal role\n"
        "2. Return the labels as a JSON array in the same order as tokens\n"
        "3. Return ONLY the JSON array, no additional text\n\n"
        "Example output format:\n"
        '["B-CAUSE", "I-CAUSE", "O", "B-EFFECT", "I-EFFECT"]'
    )

def parse_token_labels_response(response_text: str, num_tokens: int):
    """解析模型返回的 token 标签"""
    response_text = response_text.strip()
    
    # 尝试提取 JSON 数组
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if match:
        try:
            labels = json.loads(match.group())
            if isinstance(labels, list):
                # 调整长度匹配
                if len(labels) > num_tokens:
                    labels = labels[:num_tokens]
                elif len(labels) < num_tokens:
                    labels = labels + ["O"] * (num_tokens - len(labels))
                return labels, True, None
        except json.JSONDecodeError as e:
            return None, False, f"JSON解析失败: {e}"
    return None, False, "未找到JSON数组"

def ask_cd_sample(tokens):
    """调用API获取预测标签"""
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_cd_token_prompt(tokens)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a linguistic annotation expert. Respond only with valid JSON arrays."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        
        if response.status_code != 200:
            return None, False, f"API Error: {response.status_code}"
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return None, False, "No response"
            
        txt = result['choices'][0]['message']['content']
        pred_labels, success, error = parse_token_labels_response(txt, len(tokens))
        return pred_labels, success, error
        
    except Exception as e:
        return None, False, str(e)

# 自动评估前10个样本
test_size = min(10, len(ds))
print(f"\n自动评估前 {test_size} 个样本...\n")

all_true = []
all_pred = []
sample_results = []

for i in range(test_size):
    sample = ds[i]
    tokens = sample["tokens"]
    true_labels = sample["labels"]
    
    print(f"样本 {i}: 正在处理 ({len(tokens)} tokens)...")
    
    pred_labels, success, error = ask_cd_sample(tokens)
    
    if success and pred_labels:
        all_true.extend(true_labels)
        all_pred.extend(pred_labels)
        sample_results.append({
            "index": i,
            "status": "success",
            "token_count": len(tokens),
            "true_preview": true_labels[:5],
            "pred_preview": pred_labels[:5]
        })
        print(f"  ✅ 成功 | 预测前5: {pred_labels[:5]}")
    else:
        # 失败时用O填充
        all_true.extend(true_labels)
        all_pred.extend(["O"] * len(tokens))
        sample_results.append({
            "index": i,
            "status": f"failed: {error}",
            "token_count": len(tokens),
            "true_preview": true_labels[:5],
            "pred_preview": ["O"] * 5
        })
        print(f"  ❌ 失败: {error}")
    
    # 避免API限流
    time.sleep(0.5)

# 计算指标
print("\n" + "="*60)
print("EVALUATION RESULTS - 前10个样本自动评估")
print("="*60)

labels = ["O", "B-CAUSE", "I-CAUSE", "B-EFFECT", "I-EFFECT"]
accuracy = accuracy_score(all_true, all_pred)

print(f"\n总Token数: {len(all_true)}")
print(f"Token-level Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(all_true, all_pred, labels=labels, zero_division=0))

# 每个类别的准确率
print("\nPer-class Accuracy:")
for label in labels:
    mask_true = [l == label for l in all_true]
    if sum(mask_true) > 0:
        correct = sum(1 for j in range(len(all_true)) if all_true[j] == label and all_pred[j] == label)
        acc = correct / sum(mask_true)
        print(f"  {label}: {correct}/{sum(mask_true)} ({acc:.2%})")

# 显示每个样本的详细结果
print("\n" + "="*60)
print("各样本详细结果")
print("="*60)
for r in sample_results:
    print(f"\n样本 {r['index']}:")
    print(f"  状态: {r['status']}")
    print(f"  Token数: {r['token_count']}")
    print(f"  真实标签前5: {r['true_preview']}")
    print(f"  预测标签前5: {r['pred_preview']}")

# 保存结果
eval_results = {
    "model": MODEL,
    "dataset": "ChanceFocus/flare-cd",
    "task": "token-level causal annotation",
    "test_samples": test_size,
    "total_tokens": len(all_true),
    "accuracy": float(accuracy),
    "classification_report": classification_report(all_true, all_pred, labels=labels, zero_division=0, output_dict=True),
    "sample_results": sample_results,
    "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
}

eval_path = os.path.join(save_dir, f"{run_tag}_auto_10samples.json")
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)

print(f"\n评估结果已保存 -> {eval_path}")

# 对比手动和自动结果
print("\n" + "="*60)
print("对比说明")
print("="*60)
print("自动评估结果与之前手动输入的结果应该一致")
print("如果一致，说明自动评估流程正确，可以运行完整评估")

自动评估前10个样本 - FLARE-CD

自动评估前 10 个样本...

样本 0: 正在处理 (31 tokens)...
  ✅ 成功 | 预测前5: ['O', 'O', 'O', 'O', 'O']
样本 1: 正在处理 (46 tokens)...
